# **5. DQN para _cliff walking_ con Stable-Baselines3**

En [01_cliffwalking-qlearning.ipynb](../from-scratch/01_cliffwalking-qlearning.ipynb) y [02_cliffwalking-dqn.ipynb](../from-scratch/02_cliffwalking-dqn.ipynb) resolvimos el problema de [Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) con nuestras propias implementaciones de Q-Learning y DQN. En este cuaderno, usamos la implementación de [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) de [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), una librería de implementaciones fiables y probadas de los algoritmos de aprendizaje por refuerzo profundo (DRL) más comunes, construida sobre PyTorch.

SB3 no incluye métodos tabulares: DQN sustituye la tabla $Q$ por una red neuronal que aproxima $Q(s, a)$. Esto no es necesario para un problema de solo 48 estados, pero muestra que el mismo código puede aplicarse a problemas con espacios de estados mucho mayores (o continuos), donde una tabla no es viable.

_Considere el mundo de cuadrícula ("gridworld") mostrado a la derecha. Es una tarea episódica estándar, sin descuento, con estados inicial y meta, y con las acciones habituales que provocan un movimiento hacia arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="../../img/cliffwalking.png" width=500/>

Basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Disponible en [GitHub](https://github.com/DLR-RM/stable-baselines3).

Comenzamos importando las librerías necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        score += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Puntuación: ", score)

def monitor_scores(env):
    """
    Devuelve la puntuación de cada episodio registrada por Monitor, en el orden en que terminaron.

    Parámetros
    ==========
        env: entorno (o entorno vectorizado) envuelto con Monitor
    """
    # Un entorno vectorizado contiene varias copias del entorno envueltas con Monitor
    monitors = [env] if isinstance(env, Monitor) else env.envs
    scores = np.concatenate([m.get_episode_rewards() for m in monitors])
    ends = np.concatenate([np.cumsum(m.get_episode_lengths()) for m in monitors])
    return scores[np.argsort(ends)]   # Ordenar los episodios por el paso de tiempo (de cada entorno) en que terminaron

def plot_scores(scores, title, solved_score=None, window=100):
    """
    Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label="Puntuación")
    plt.plot(np.arange(len(scores)), avg, label=f"Puntuación media (últimos {window})")
    if solved_score is not None:
        plt.axhline(solved_score, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Puntuación")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

def print_policy(actions):
    """
    Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(["↑", "→", "↓", "←"])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = "C"   # Celdas del acantilado
    grid[3, 11] = "T"     # Meta
    print("\n".join(" ".join(row) for row in grid))

def plot_values(V):
    """
    Representa la función de valor de estado sobre la cuadrícula de 4 x 12 celdas.

    Parámetros
    ==========
        V (array_like): valor de cada uno de los 48 estados
    """
    V = np.reshape(V, (4, 12))
    fig = plt.figure(figsize=(15, 5))
    ax = fig.add_subplot(111)
    ax.imshow(V, cmap="cool")
    for (j, i), label in np.ndenumerate(V):
        ax.text(i, j, f"{label:.1f}", ha="center", va="center", fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title("Función de valor de estado")
    plt.show()

## **5.1. Entorno**

El estado es el índice de la celda en la que se encuentra el agente (de $0$ a $47$) y hay $4$ acciones posibles: `0` (arriba), `1` (derecha), `2` (abajo) y `3` (izquierda). SB3 codifica automáticamente este estado discreto como un vector _one-hot_ de tamaño $48$ antes de pasarlo a la red Q.

Caer por el acantilado no termina el episodio, así que una mala política puede caminar indefinidamente. Limitamos la longitud de los episodios a $100$ pasos con el parámetro `max_episode_steps`: al alcanzar ese límite, el episodio se trunca (`truncated = True`).

In [ ]:
env = gym.make("CliffWalking-v1", render_mode="rgb_array", max_episode_steps=100)

print(env.observation_space)
print(env.action_space)

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

La función `show_episode()` muestra un episodio como una animación, junto con su **puntuación**, es decir, la suma de las recompensas obtenidas a lo largo del episodio. Veamos, por ejemplo, un episodio de un agente que elige acciones al azar:

In [ ]:
show_episode(env, lambda state: env.action_space.sample())

## **5.2. Algoritmo DQN**

La clase [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) incluye la red Q, la red objetivo (_target network_), la memoria de experiencias (_replay buffer_) y la estrategia de exploración $\epsilon$-_greedy_. En lugar de un número de episodios, SB3 entrena durante un número dado de pasos de tiempo (`total_timesteps`).

SB3 necesita que el entorno de entrenamiento esté envuelto en un [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) para registrar la puntuación de cada episodio. Usaremos estos valores para representar la curva de aprendizaje.

In [ ]:
env_dqn = Monitor(gym.make("CliffWalking-v1", max_episode_steps=100))

model_dqn = DQN(
    "MlpPolicy",                        # Red Q: perceptrón multicapa
    env_dqn,
    learning_rate=1e-3,
    batch_size=64,
    buffer_size=50_000,
    learning_starts=1000,               # Pasos con acciones aleatorias antes de empezar a aprender
    gamma=0.99,
    target_update_interval=500,         # Copiar la red Q en la red objetivo cada 500 pasos
    train_freq=4,                       # Actualizar la red Q cada 4 pasos
    exploration_fraction=0.3,           # Epsilon decrece linealmente durante el primer 30% del entrenamiento...
    exploration_final_eps=0.05,         # ...hasta este valor
    policy_kwargs=dict(net_arch=[64, 64]),
    seed=0,
    verbose=0,
)

model_dqn.learn(total_timesteps=50_000, progress_bar=True)
model_dqn.save("cliffwalking-sb3-dqn")   # Guardar el agente entrenado

plot_scores(monitor_scores(env_dqn), "DQN")

La política óptima (caminar por el borde del acantilado) obtiene una puntuación de $-13$. Evaluemos el agente con la política _greedy_ (`deterministic=True`). Como el entorno es determinista, un único episodio es suficiente.

In [ ]:
mean_score, std_score = evaluate_policy(model_dqn, Monitor(gym.make("CliffWalking-v1", max_episode_steps=100)), n_eval_episodes=1, deterministic=True)
print(f"DQN\tPuntuación de la política greedy: {mean_score:.2f}")

## **5.3. Política aprendida y función de valor de estado**

La política aprendida es la siguiente. La obtenemos pidiendo al modelo la acción _greedy_ (`predict`) en cada uno de los 48 estados.

In [ ]:
states = np.arange(48)
actions, _ = model_dqn.predict(states, deterministic=True)

print_policy(actions)

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$. Aquí los valores $Q(s, a)$ no se leen de una tabla, sino que los calcula la red Q (`model_dqn.q_net`).

Nótese que hemos entrenado con $\gamma = 0.99$ en lugar de $\gamma = 1$, por lo que los valores de los estados lejanos a la meta son ligeramente mayores (menos negativos) que los óptimos.

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
obs, _ = model_dqn.policy.obs_to_tensor(states)   # Codificación one-hot de los 48 estados
with torch.no_grad():
    Q = model_dqn.q_net(obs).numpy()              # Valores Q de los 48 estados, forma (48, 4)

V = Q.max(axis=1)
V[37:] = 0   # Las celdas del acantilado y de la meta nunca se visitan
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4, 12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## **5.4. Visualizar el agente entrenado**

Cargamos el agente entrenado y lo ejecutamos durante un episodio con la política _greedy_ (`deterministic=True`):

In [ ]:
# Cargar el agente entrenado desde `cliffwalking-sb3-dqn.zip`
model_load = DQN.load("cliffwalking-sb3-dqn")

show_episode(env, lambda state: int(model_load.predict(state, deterministic=True)[0]))

¡Nuestro agente ha aprendido a cruzar el acantilado con unas pocas líneas de código!

En este cuaderno, hemos resuelto Cliff Walking con la implementación de DQN de Stable-Baselines3: el agente, la red neuronal, la memoria de experiencias y el bucle de entrenamiento ya están implementados, y solo tenemos que elegir el algoritmo y sus hiperparámetros.

En el próximo cuaderno, aplicaremos Stable-Baselines3 a CartPole y compararemos dos algoritmos: DQN y PPO.